In [ ]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime

all_records = []

start_year = datetime.now().year - 5
end_year = datetime.now().year

#GETTING DATA FROM API

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

for year in range(start_year, end_year + 1):

    if year == start_year:
        start_month = datetime.now().month
    else:
        start_month = 1

    if year == end_year:
        end_month = datetime.now().month
    else:
        end_month = 12

    for month in range(start_month, end_month + 1):

        start_date = f"{year}-{month:02d}-01"

        if month == 12:
            end_date = f"{year + 1}-01-01"
        else:
            end_date = f"{year}-{month + 1:02d}-01"

        params = {
            "format": "geojson",
            "starttime": start_date,
            "endtime": end_date,
            "minmagnitude": 3
        }

        response = requests.get(url, params=params)

        if response.status_code != 200:
            print(f"Failed for {start_date}: {response.text[:200]}")
            continue

        try:
            data = response.json()
        except Exception as e:
            print(f"JSON error for {start_date}: {e}")
            continue

        for f in data["features"]:

            p = f["properties"]
            g = f["geometry"]["coordinates"]

            all_records.append({
                "id": f.get("id"),

                # CONVERT DATETIME FIELDS FOR TIME/UPDATED
                "time": pd.to_datetime(
                    p.get("time"), unit="ms", errors="coerce"
                ),

                "updated": pd.to_datetime(
                    p.get("updated"), unit="ms", errors="coerce"
                ),

                "latitude": g[1] if g else None,
                "longitude": g[0] if g else None,
                "depth_km": g[2] if g else None,

                "mag": p.get("mag"),
                "magType": p.get("magType"),
                "place": p.get("place"),
                "status": p.get("status"),
                "tsunami": p.get("tsunami"),
                "sig": p.get("sig"),
                "net": p.get("net"),
                "nst": p.get("nst"),
                "dmin": p.get("dmin"),
                "rms": p.get("rms"),
                "gap": p.get("gap"),
                "magError": p.get("magError"),
                "depthError": p.get("depthError"),
                "magNst": p.get("magNst"),


                "types": p.get("types"),
                "ids": p.get("ids"),
                "sources": p.get("sources"),
                "type": p.get("type"),

                "code": p.get("code"),       
                "alert": p.get("alert"),
                "felt": p.get("felt"),
                "cdi": p.get("cdi"),
                "mmi": p.get("mmi"),
            })

        print(
            f"{start_date} completed | "
            f"Total records: {len(all_records)}"
        )

# CREATE DataFrame

df = pd.DataFrame(all_records)

print("Original shape:", df.shape)





# Other string fields
text_cols = [
    "magType",
    "status",
    "type",
    "net",
    "sources",
    "types"
]

for col in text_cols:
    df[col] = df[col].str.strip().str.lower()



#  EXTRACT COUNTRY FROM PLACE

df["country"] = (
    df["place"]
    .str.extract(r",\s*([^,]+)$")[0].str.strip().str.lower()
)
df["country"] = df["country"].fillna("unknown")

#  CONVERT NUMERIC FIELDS

numeric_cols = [
    "mag","depth_km","nst","dmin","rms","gap","magError",
    "depthError","magNst","sig","felt","cdi","mmi"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# HANDLE MISSING VALUES

df["alert"] = (
    df["alert"].fillna("unknown").str.strip().str.lower()
)

"""
df["magError"] = df["magError"].fillna(0)
df["depthError"] = df["depthError"].fillna(0)
df["magNst"] = df["magNst"].fillna(0)
"""
df["felt"] = df["felt"].fillna(0)


# Fill missing numeric values with median
median_cols = [
    "cdi", "mmi","nst","dmin","rms","gap",
    "magError","depthError","magNst"
]

for col in median_cols:
    df[col] = df[col].fillna(df[col].median())


# DERIVED COLUMNS

# Date-based columns
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["day_of_week"] = df["time"].dt.day_name()

# Depth classification
df["depth_category"] = np.where(
    df["depth_km"] < 50,
    "shallow",
    "deep"
)

# Strong earthquake
df["strong_earthquake"] = np.where(
    df["mag"] >= 6,
    "strong",
    "not strong"
)

# Destructive earthquake
df["destructive_earthquake"] = np.where(
    df["mag"] >= 7,
    "destructive",
    "not destructive"
)

print(df.isnull().sum())

2021-09-01 completed | Total records: 1688
2021-10-01 completed | Total records: 3224
2021-11-01 completed | Total records: 4767
2021-12-01 completed | Total records: 6682


In [3]:
print(df.head(2))

           id                    time                 updated  latitude  \
0  us6000fql6 2021-09-30 21:27:45.271 2025-12-22 22:17:17.129   25.4511   
1  us6000fqkg 2021-09-30 20:33:01.221 2021-12-04 14:28:39.040   25.5571   

   longitude  depth_km  mag magType  \
0  -109.4361      10.0  4.4      mb   
1  -109.6688      10.0  4.3      mb   

                                             place    status  ...  cdi    mmi  \
0  36 km SSW of Campo Pesquero el Colorado, Mexico  reviewed  ...  3.4  3.366   
1  41 km WSW of Campo Pesquero el Colorado, Mexico  reviewed  ...  3.1  3.366   

  country  year  month  day  day_of_week  depth_category  strong_earthquake  \
0  mexico  2021      9   30     Thursday         shallow         not strong   
1  mexico  2021      9   30     Thursday         shallow         not strong   

   destructive_earthquake  
0         not destructive  
1         not destructive  

[2 rows x 37 columns]


In [4]:
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())

(103957, 37)
id                                   str
time                      datetime64[ms]
updated                   datetime64[ms]
latitude                         float64
longitude                        float64
depth_km                         float64
mag                              float64
magType                              str
place                                str
status                               str
tsunami                            int64
sig                                int64
net                                  str
nst                              float64
dmin                             float64
rms                              float64
gap                              float64
magError                         float64
depthError                       float64
magNst                           float64
types                                str
ids                                  str
sources                              str
type                                 str
cod

In [6]:
# Save cleaned data as CSV
df.to_csv("earthquakes_cleaned.csv", index=False)

print("Cleaned data saved successfully!")

Cleaned data saved successfully!


In [7]:
print(df[["time", "updated"]].dtypes)

time       datetime64[ms]
updated    datetime64[ms]
dtype: object


In [9]:
for col in ["magError", "depthError", "magNst"]:
    print("\nColumn:", col)
    print("Unique values:", df[col].unique())
    print("Number of unique values:", df[col].nunique(dropna=False))


Column: magError
Unique values: [nan]
Number of unique values: 1

Column: depthError
Unique values: [nan]
Number of unique values: 1

Column: magNst
Unique values: [nan]
Number of unique values: 1


In [10]:
pip install sqlalchemy pymysql


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from sqlalchemy import create_engine

In [7]:
username = "root"
password = "DivyaMysql11"
host = "localhost"
database = "earthquake_db"
# Create SQLAlchemy engine
engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}/{database}"
)
print("Engine created successfully!")

Engine created successfully!


In [13]:
#  CREATE TABLE AND INSERT DATA 
df.to_sql( "earthquakes", con=engine,
           if_exists="replace", index=False )
print("Earthquake data inserted into MySQL successfully!")


Earthquake data inserted into MySQL successfully!


In [14]:
#Magnitude & Depth 
#1. Top 10 strongest earthquakes (mag). 
query = """
Select * from earthquakes
order by mag DESC
limit 10
"""

result = pd.read_sql(query, engine)

print(result)

           id                time             updated  latitude  longitude  \
0  us6000qw60 2025-07-29 23:24:52 2026-07-31 16:16:07   52.4948   160.2395   
1  us7000qx2g 2025-09-18 18:58:15 2025-12-06 16:41:30   53.1426   160.7206   
2  us6000tkt2 2026-08-14 21:58:22 2026-09-10 14:29:01   -8.3514   121.3478   
3  us7000srb1 2026-06-07 23:37:42 2026-09-09 10:16:11    5.5994   125.0560   
4  us6000jllz 2023-02-06 01:17:34 2026-04-13 23:32:30   37.2256    37.0143   
5  us6000kd0n 2023-05-19 02:57:03 2025-08-15 23:01:19  -23.2063   170.7423   
6  us7000pn9s 2025-03-28 06:20:53 2026-07-17 19:33:31   22.0110    95.9363   
7  us7000pcdl 2025-02-08 23:23:15 2025-04-26 15:51:50   17.6506   -82.3950   
8  us7000lff4 2023-12-02 14:37:04 2026-06-27 07:18:57    8.5266   126.4161   
9  us6000kawn 2023-05-10 16:02:00 2023-07-22 19:16:37  -15.6278  -174.4925   

   depth_km  mag magType                                              place  \
0    35.000  8.8     mww        2025 Kamchatka Peninsula, Russ

In [ ]:
# 2.Top 10 deepest earthquakes (depth_km). 
query1 = """
Select  id,time,place,depth_km,mag,magType
From earthquakes
order by depth_km DESC
limit 10;
"""

result1 = pd.read_sql(query1, engine)

print(result1)

           id                time                        place  depth_km  mag  \
0  us6000sp3p 2026-04-02 16:59:18  206 km ENE of Sola, Vanuatu   683.578  4.0   
1  us6000k2db 2023-04-01 18:09:17  208 km ENE of Sola, Vanuatu   681.238  4.0   
2  us7000kxdn 2023-09-18 15:35:27               Vanuatu region   675.265  4.2   
3  us6000mivr 2024-03-08 22:42:24                  Fiji region   671.043  4.2   
4  us6000rk66 2025-10-29 17:07:34   205 km ESE of Levuka, Fiji   669.556  4.8   
5  us7000q1jk 2025-05-10 02:03:17     299 km E of Levuka, Fiji   667.237  4.2   
6  us7000s0st 2026-02-14 08:27:48   205 km ESE of Levuka, Fiji   667.197  4.3   
7  us6000ta9m 2026-07-05 14:22:25    274 km SE of Levuka, Fiji   665.993  5.7   
8  us6000tknp 2026-08-10 16:50:10   178 km NE of Sola, Vanuatu   665.326  4.0   
9  us7000he64 2022-06-01 18:41:01   279 km ESE of Labasa, Fiji   664.700  4.3   

  magType  
0      mb  
1      mb  
2      mb  
3      mb  
4      mb  
5      mb  
6      mb  
7     mwb  


In [ ]:
# shallow earthquakes < 50 km and mag > 7.5. 
query3 = """
select id, time, place, depth_km, mag
from earthquakes
where depth_category = 'shallow' and mag > 7.5;
"""

result3 = pd.read_sql(query3, engine)

print(result3)

            id                time  \
0   us7000i9bw 2022-09-19 18:05:08   
1   us6000jllz 2023-02-06 01:17:34   
2   us6000kd0n 2023-05-19 02:57:03   
3   us7000lff4 2023-12-02 14:37:04   
4   us7000pcdl 2025-02-08 23:23:15   
5   us7000pn9s 2025-03-28 06:20:53   
6   us6000qw60 2025-07-29 23:24:52   
7   us7000qx2g 2025-09-18 18:58:15   
8   us6000rgf4 2025-10-10 20:29:20   
9   us6000rtdt 2025-12-08 14:15:10   
10  us6000tkt2 2026-08-14 21:58:22   

                                                place  depth_km  mag  
0                      35 km SSW of Aguililla, Mexico    26.943  7.6  
1   Pazarcik earthquake, Kahramanmaras earthquake ...    10.000  7.8  
2                    southeast of the Loyalty Islands    18.053  7.7  
3                       19 km E of Gamut, Philippines    40.000  7.6  
4           210 km SSW of George Town, Cayman Islands    14.326  7.6  
5           2025 Mandalay, Burma (Myanmar) Earthquake    10.000  7.7  
6         2025 Kamchatka Peninsula, Russia Ear

In [ ]:
#5. Average magnitude per magnitude type (magType). 
query4=""" 
select magType, AVG(mag) 
from earthquakes
group by magType;
"""
result4=pd.read_sql(query4,engine)

print(result4)

       magType  AVG(mag)
0           mb  4.416871
1           ml  3.269010
2           md  3.445824
3          mww  5.355392
4          mwr  4.317558
5           mw  3.914637
6          mwb  5.762963
7        mb_lg  3.195098
8          mlg  3.300000
9          mlr  3.487500
10       ms_20  5.800000
11         mlv  3.469288
12         mwc  7.000000
13  ml(texnet)  3.367442
14       ms_vx  4.650000
15         mwp  5.250000
16          mh  4.100000


In [ ]:
#Time Analysis 
#6. Year with most earthquakes.
query5="""
select year,count(*) as earthquake_count 
from earthquakes
group by year
order by earthquake_count desc
"""

result5=pd.read_sql(query5,engine)

print(result5)

   year  earthquake_count
0  2025             22819
1  2023             20830
2  2022             20208
3  2024             18659
4  2026             14734
5  2021              6682


In [ ]:
#7. Month with highest number of earthquakes. 
query6=""" 
select month,count(*) AS earthquake_count
from earthquakes
group by month
order by earthquake_count DESC
limit 5;
"""
result6=pd.read_sql(query6,engine)

print(result6)

   month  earthquake_count
0      7              9987
1     12              9675
2      8              8982
3      1              8864
4      4              8753


In [ ]:
#8. Day of week with most earthquakes. 
query7=""" 
select  day_of_week,count(*) as earthquake_count
from earthquakes
group by day_of_week
order by earthquake_count desc
limit 5;
"""

result7=pd.read_sql(query7,engine)

print(result7)

  day_of_week  earthquake_count
0     Tuesday             15106
1   Wednesday             15061
2      Monday             14975
3      Sunday             14892
4      Friday             14823


In [ ]:
#9. Count of earthquakes per hour of day. 

query8="""
select hour(time) as hour,count(*) as earthquake_count
from earthquakes
group by hour
order by earthquake_count desc
limit 5
"""

result8=pd.read_sql(query8,engine)

print(result8)

   hour  earthquake_count
0     3              4589
1     4              4582
2    21              4579
3     1              4577
4    22              4520


In [ ]:
#10.   Most active reporting network (net). 
query9="""
select net,count(*) as earthuake_count
from earthquakes
group by net
order by earthuake_count desc
"""

result9=pd.read_sql(query9,engine)

print(result9)

       net  earthuake_count
0       us            91171
1       pr             5515
2       ak             3222
3       tx              916
4       hv              840
5       nc              836
6       ci              759
7       nn              402
8       uu               99
9       ok               72
10      uw               53
11      nm               21
12      se               16
13      av                5
14  iscgem                3
15      mb                1
16      ew                1


In [ ]:
#11.  Top 5 places with highest casualties. 
query10="""
select place,SUM(felt) AS total_felt
from earthquakes
where felt > 0
group by place
order by total_felt desc
limit 5;
"""
result10=pd.read_sql(query10,engine)
print(result10)

                                   place  total_felt
0  2024 Tewksbury, New Jersey Earthquake    184673.0
1       21 km SE of Greenback, Tennessee     45935.0
2                   5 km S of Julian, CA     43198.0
3          9 km SE of York Harbor, Maine     42440.0
4           1 km SE of Boulder Creek, CA     33078.0


In [ ]:
#13.  Average economic loss by alert level. 

query11="""
select alert,count(*) AS earthquake_count
from earthquakes
Where alert != 'unknown'
group by alert
order by earthquake_count DESC;
"""

result11=pd.read_sql(query11,engine)
print(result11)

    alert  earthquake_count
0   green              3957
1  yellow               125
2     red                25
3  orange                22


In [ ]:
#Event Type & Quality Metrics 
#14.  Count of reviewed vs automatic earthquakes (status).

query12="""
select status,count(*) as earthquake_count
from earthquakes
group by status
"""
result12=pd.read_sql(query12,engine)

print(result12)

      status  earthquake_count
0   reviewed            103811
1  automatic               121


In [ ]:
#15.  Count by earthquake type (type). 
query13 = """
select  type,count(*) AS earthquake_count
from earthquakes
group by type
order by earthquake_count DESC;
"""

result13 = pd.read_sql(query13, engine)

print(result13)

                     type  earthquake_count
0              earthquake            103153
1        mining explosion               741
2       volcanic eruption                12
3               ice quake                 9
4               landslide                 6
5             other event                 5
6            quarry blast                 2
7  experimental explosion                 2
8           mine collapse                 1
9               explosion                 1


In [ ]:
#16.  Number of earthquakes by data type (types). 
query14=""" 
select types,count(*) as earthquake_types_count
from earthquakes
group by types
"""

result14=pd.read_sql(query14,engine)

print(result14)

                                                 types  earthquake_types_count
0                             ,dyfi,origin,phase-data,                    7512
1                                  ,origin,phase-data,                   78851
2    ,dyfi,focal-mechanism,nearby-cities,origin,pha...                     420
3                    ,moment-tensor,origin,phase-data,                    1314
4                         ,origin,phase-data,shakemap,                    2458
..                                                 ...                     ...
542  ,dyfi,ground-failure,impact-text,internal-mome...                       1
543  ,dyfi,event-sequence,finite-fault,general-text...                       2
544  ,dyfi,event-sequence,moment-tensor,oaf,origin,...                       1
545  ,dyfi,event-sequence,impact-link,moment-tensor...                       1
546  ,dyfi,moment-tensor,origin,phase-data,shake-al...                       1

[547 rows x 2 columns]


In [ ]:
#18.  Events with high station coverage (nst > threshold).
query15=""" 
select id,place, mag,nst
from earthquakes
where nst > 50
order by nst desc;
 """
result15=pd.read_sql(query15,engine)

print(result15)

               id                                              place   mag  \
0      us6000m12f                          11 km W of Anamizu, Japan  5.40   
1      us6000qzfl                  120 km ENE of Ozernovskiy, Russia  5.00   
2      us7000rluk                        111 km N of Yakutat, Alaska  5.70   
3      us7000pvtr                            Macquarie Island region  6.80   
4      usd001097k  49 km WNW of San Antonio de los Cobres, Argentina  5.50   
...           ...                                                ...   ...   
26335  us6000tjhi                      39 km SW of Waisai, Indonesia  4.50   
26336  us6000tjer                         southeast of Easter Island  5.20   
26337  us6000tjel                          central East Pacific Rise  5.10   
26338  us6000timi                          153 km E of Miyako, Japan  4.60   
26339  nc75412157                            31 km NNW of Covelo, CA  4.41   

         nst  
0      619.0  
1      566.0  
2      516.0  
3  

In [5]:
#print("uniqu",df["tsunami"].unique())
#Tsunamis & Alerts 
#19.  Number of tsunamis triggered per year. 

query16=""" 
select year,count(*) as eathquake_count_tunami
from earthquakes
where tsunami=1
group by year
order by year
"""
result16=pd.read_sql(query16,engine)

print(result16)


   year  eathquake_count_tunami
0  2021                      39
1  2022                     136
2  2023                     119
3  2024                     114
4  2025                     142
5  2026                      40


In [ ]:
#20.Count earthquakes by alert levels (red, orange, etc.).

query17=""" 
select alert,count(*) as earthquae_alert_count
from earthquakes
group by alert
"""
result17=pd.read_sql(query17,engine)

print(result17)

     alert  earthquae_alert_count
0  unknown                  99803
1    green                   3957
2   yellow                    125
3      red                     25
4   orange                     22


In [18]:
#Seismic Pattern & Trends Analysis.             
#21.Find the top 5 countries with the highest average magnitude of earthquakes in past 5 years 

#print(df["country"].unique())

query18=""" 
select country,avg(mag) as avg_magnitude
from earthquakes
where country !="unknown"
group by country
order by avg_magnitude desc
limit 5
"""

result18=pd.read_sql(query18,engine)

print(result18)

                             country  avg_magnitude
0                  russia earthquake           8.10
1         burma (myanmar) earthquake           7.70
2  kahramanmaras earthquake sequence           7.65
3                   japan earthquake           7.25
4                  alaska earthquake           7.25


In [15]:
#print(df["depth_category"].unique())
#22.Find countries that have experienced both shallow and deep earthquakes within the same month. 

query19="""
select country,year,month
from earthquakes
where country != 'unknown'
group by country, year, month
having sum(depth_km < 70) > 0 and sum(depth_km > 300) > 0
limit 10;
"""

result19=pd.read_sql(query19,engine)
print(result19)



                    country  year  month
0              japan region  2021      9
1                     tonga  2021      9
2               philippines  2021      9
3                     japan  2021      9
4  northern mariana islands  2021      9
5                    russia  2021      9
6               timor leste  2021      9
7                      fiji  2021      9
8                      fiji  2021     10
9                     japan  2021     10


In [ ]:
#23 Compute the year-over-year growth rate in the total number of earthquakes globally.

query20 = """
SELECT
    year,
    earthquake_count,
    LAG(earthquake_count) OVER (ORDER BY year) AS previous_year_count,
    ROUND(
        (earthquake_count - LAG(earthquake_count) OVER (ORDER BY year))
        / LAG(earthquake_count) OVER (ORDER BY year) * 100,
        2
    ) AS growth_rate
FROM (
    SELECT
        year,
        COUNT(*) AS earthquake_count
    FROM earthquakes
    GROUP BY year
) AS yearly_data
ORDER BY year;
"""

result20 = pd.read_sql(query20, engine)
print(result20)

   year  earthquake_count  previous_year_count  growth_rate
0  2021              6682                  NaN          NaN
1  2022             20208               6682.0       202.42
2  2023             20830              20208.0         3.08
3  2024             18659              20830.0       -10.42
4  2025             22819              18659.0        22.29
5  2026             14759              22819.0       -35.32


In [ ]:
#24. List the 3 most seismically active regions by combining both frequency and average magnitude. 

query21="""
select country,count(*) as earthquake_count,avg(mag) as avg_magnitude
from earthquakes
where country !="unknown"
group by country
order by earthquake_count desc, avg_magnitude desc
limit 3
"""

result21=pd.read_sql(query21,engine)
print(result21)

     country  earthquake_count  avg_magnitude
0     alaska             13277       3.467291
1  indonesia              8355       4.502501
2     russia              6151       4.483499


In [8]:
#Depth, Location & Distance-Based  Analysis. 
#25. For each country, calculate the average depth of earthquakes within ±5° latitude range of the equator. 

query22="""
select country,avg(depth_km) as avg_depth
from earthquakes
where country !="unknown" and latitude between -5 and 5    
group by country
order by avg_depth desc
"""

result22=pd.read_sql(query22,engine)
print(result22)


                             country   avg_depth
0                        philippines  106.173610
1                   papua new guinea   71.696617
2                               peru   63.622641
3                            ecuador   62.982260
4                          indonesia   61.789330
5                           colombia   47.256295
6                       congo-uganda   14.775000
7                          venezuela   11.127000
8                             brazil   10.834000
9                           malaysia   10.762500
10                            uganda   10.081739
11                            rwanda   10.000000
12                          tanzania   10.000000
13                             palau   10.000000
14                           burundi   10.000000
15                          ethiopia   10.000000
16                             gabon   10.000000
17                   kiribati region   10.000000
18                       south sudan   10.000000
19                  

In [11]:
#26. Identify countries having the highest ratio of shallow to deep earthquakes. 

query23="""
select country, sum(depth_category = 'shallow') as shallow_count,
    sum(depth_category = 'deep') as deep_count,
    sum(depth_category = 'shallow') / sum(depth_category = 'deep') as shallow_to_deep_ratio
from earthquakes
where country != "unknown"
group by country
having sum(depth_category = 'deep') > 0
order by shallow_to_deep_ratio desc
limit 10;
"""

result23=pd.read_sql(query23,engine)
print(result23)

      country  shallow_count  deep_count  shallow_to_deep_ratio
0        iran          869.0         7.0               124.1429
1      panama          189.0         2.0                94.5000
2        utah           78.0         1.0                78.0000
3      canada          277.0         4.0                69.2500
4  kyrgyzstan          126.0         2.0                63.0000
5       china         1275.0        23.0                55.4348
6     morocco           49.0         1.0                49.0000
7      turkey          923.0        23.0                40.1304
8       nepal          140.0         4.0                35.0000
9      cyprus           34.0         1.0                34.0000


In [ ]:
#27. Find the average magnitude difference between earthquakes with tsunami alerts and those without. 

query24=""" 
select avg(case when tsunami = 1 then mag else null end) AS avg_mag_with_tsunami,
       avg(case when tsunami = 0 then mag else null end) AS avg_mag_without,
       round(avg(case when tsunami = 1 then mag else null end) - 
          avg(case when tsunami = 0 then mag else null end), 2) AS avg_mag_difference 
from earthquakes;
"""

result24=pd.read_sql(query24,engine)
print(result24)


   avg_mag_with_tsunami  avg_mag_without  avg_mag_difference
0              5.383231         4.243074                1.14


In [14]:
#28. Using the gap and rms columns, identify events with the lowest data reliability (highest average error margins). 

query25="""
select id, country, gap, rms, (gap + rms) / 2 AS avg_error_margin
from earthquakes  
where country != "unknown"
order by avg_error_margin desc
limit 20;
"""

result25=pd.read_sql(query25,engine)
print(result25)



              id              country     gap     rms  avg_error_margin
0     pr71519528  u.s. virgin islands  359.00  0.1400         179.57000
1     nn00897599               nevada  358.18  0.1601         179.17005
2   pr2024051000             anguilla  358.00  0.1600         179.08000
3     pr71508583             anguilla  356.00  0.2200         178.11000
4   pr2022227000  u.s. virgin islands  353.00  0.6100         176.80500
5     pr71489653   dominican republic  352.00  0.1800         176.09000
6   pr2022289000  u.s. virgin islands  352.00  0.0700         176.03500
7   pr2022311006  u.s. virgin islands  351.00  0.1700         175.58500
8     us6000k2l1               alaska  350.00  0.2400         175.12000
9     av91047508               alaska  350.00  0.1000         175.05000
10  pr2022230002  u.s. virgin islands  350.00  0.0300         175.01500
11  pr2022031001  u.s. virgin islands  349.00  0.6200         174.81000
12  pr2021274004  u.s. virgin islands  349.00  0.5000         17

In [15]:
#30. Determine the regions with the highest frequency of deep-focus earthquakes (depth > 300 km). 

query26=""" 
select country, count(*) as deep_earthquake_count
from earthquakes
where depth_km > 300 and country != "unknown"
group by country
order by deep_earthquake_count desc
limit 10;
"""

result26=pd.read_sql(query26,engine)
print(result26) 

                    country  deep_earthquake_count
0                      fiji                   1082
1                     tonga                    555
2                 indonesia                    252
3              japan region                    233
4               timor leste                    194
5         wallis and futuna                    137
6                     japan                    120
7  northern mariana islands                    116
8               philippines                    104
9                    russia                     92
